# 📓 Day 2: Retrieval Optimization (Top-K, Hybrid Search & Thresholds)
## VERA (Verified Evidence Retrieval Assistant)

**Objective**: Benchmark Top-K, test Dense vs. BM25 Hybrid retrieval, and verify similarity thresholds.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.embeddings.embedder import MedicalEmbedder
from src.embeddings.vector_store import VectorStoreManager
from src.retrieval.semantic_search import SemanticSearchEngine
from src.retrieval.hybrid_retriever import HybridRetriever
from src.utils.helpers import load_json

print("Imports loaded!")

Imports loaded!


### 1. Initialize Retrieval Engines

In [2]:
chunks_data = load_json("../data/processed/chunk_catalog.json")
embedder = MedicalEmbedder()
vector_store = VectorStoreManager(persist_dir="../data/vector_db", embedder=embedder)

dense_engine = SemanticSearchEngine(vector_store, similarity_threshold=0.60)
hybrid_engine = HybridRetriever(vector_store, all_chunks=chunks_data, dense_weight=0.7, bm25_weight=0.3)
print("Engines initialized!")

2026-08-16 21:52:49 | INFO     | src.embeddings.embedder:59 - Loading Local Model: 'BAAI/bge-small-en-v1.5' on device 'cpu'...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-16 21:52:54 | SUCCESS  | src.embeddings.embedder:61 - Model 'BAAI/bge-small-en-v1.5' loaded successfully (dim=384)


d:\AI Hackathon\New data\src\embeddings\embedder.py:61: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  logger.success(f"Model '{self.model_name}' loaded successfully (dim={self.model.get_sentence_embedding_dimension()})")


2026-08-16 21:52:55 | INFO     | src.embeddings.vector_store:44 - VectorStoreManager connected to ChromaDB collection: 'vera_clinical_guidelines' (Current count: 188)
2026-08-16 21:52:55 | INFO     | src.retrieval.hybrid_retriever:34 - Initialized BM25 index with 94 documents.
Engines initialized!


### 2. Compare Dense vs. Hybrid Retrieval

In [3]:
query = "What is the dosing and delivery method of nusinersen (Spinraza)?"

print("=== DENSE VECTOR RETRIEVAL ===")
dense_res = dense_engine.retrieve(query, top_k=3)
for r in dense_res:
    print(f"Score: {r['similarity_score']} | Doc: {r['metadata']['doc_name']} (Page {r['metadata']['page_number']})")

print("\n=== HYBRID (DENSE + BM25) RETRIEVAL ===")
hybrid_res = hybrid_engine.retrieve(query, top_k=3)
for r in hybrid_res:
    print(f"Score: {r['similarity_score']} | Doc: {r['metadata']['doc_name']} (Page {r['metadata']['page_number']})")

=== DENSE VECTOR RETRIEVAL ===
2026-08-16 21:52:55 | INFO     | src.retrieval.semantic_search:14 - Retrieving top 3 results for query: 'What is the dosing and delivery method of nusinersen (Spinraza)?'
2026-08-16 21:52:55 | INFO     | src.retrieval.semantic_search:25 - Retrieved 3 valid chunks (out of 3 candidates).
Score: 0.7023 | Doc: IJMS_2023_SMA_Past_Present_Future_Review.pdf (Page 18)
Score: 0.7023 | Doc: IJMS_2023_SMA_Past_Present_Future_Review.pdf (Page 18)
Score: 0.6949 | Doc: ClinPediatr_2023_SMA_Treatment_Best_Practices.pdf (Page 5)

=== HYBRID (DENSE + BM25) RETRIEVAL ===
2026-08-16 21:52:55 | INFO     | src.retrieval.hybrid_retriever:87 - Hybrid retrieval completed: returned top 3 chunks.
Score: 0.7695 | Doc: IJMS_2023_SMA_Past_Present_Future_Review.pdf (Page 18)
Score: 0.7222 | Doc: ClinPediatr_2023_SMA_Treatment_Best_Practices.pdf (Page 5)
Score: 0.7585 | Doc: ClinPediatr_2023_SMA_Treatment_Best_Practices.pdf (Page 4)


### 3. Log Retrieval Scores Across Test Queries

In [4]:
test_queries = [
    "How does long-read sequencing detect complex chromosomal rearrangements?",
    "What are the differences between SMN1 and SMN2 genes?",
    "What is the role of newborn screening in SMA outcomes?"
]

for q in test_queries:
    res = hybrid_engine.retrieve(q, top_k=2)
    print(f"\nQuery: '{q}'")
    print(f"Top Result Doc: {res[0]['metadata']['doc_name']} | Section: {res[0]['metadata']['section']}")

2026-08-16 21:52:55 | INFO     | src.retrieval.hybrid_retriever:87 - Hybrid retrieval completed: returned top 2 chunks.

Query: 'How does long-read sequencing detect complex chromosomal rearrangements?'
Top Result Doc: GenomeResearch_2024_LongRead_Chromosomal_Rearrangements.pdf | Section: General Overview
2026-08-16 21:52:55 | INFO     | src.retrieval.hybrid_retriever:87 - Hybrid retrieval completed: returned top 2 chunks.

Query: 'What are the differences between SMN1 and SMN2 genes?'
Top Result Doc: IJMS_2023_SMA_Past_Present_Future_Review.pdf | Section: General Overview
2026-08-16 21:52:55 | INFO     | src.retrieval.hybrid_retriever:87 - Hybrid retrieval completed: returned top 2 chunks.

Query: 'What is the role of newborn screening in SMA outcomes?'
Top Result Doc: medRxiv_2024_SMA_Missed_Diagnoses_Sequencing.pdf | Section: General Overview
